# A Generalised Data-Driven Shoreline Model Trained on the South-East Australian Coastline

**Kit Calcraft<sup>1</sup>, Joshua A. Simmons<sup>2</sup>, Lucy A. Marshall<sup>3</sup>, Kristen D. Splinter<sup>1</sup>**  
<sup>1</sup>Water Research Laboratory, School of Civil and Environmental Engineering, UNSW Sydney  
<sup>2</sup>ARC Training Centre in Data Analytics for Resources and Environments (DARE), Sydney, NSW, Australia  
<sup>3</sup>Faculty of Engineering, The University of Sydney  

---

### Workflow Description
This notebook provides a complete workflow for training, loading, and evaluating a generalised shoreline forecasting model.  
- **Configuration**: Model hyperparameters and data settings are loaded from YAML files.  
- **Data Preparation**: Dataloaders are constructed to manage training and holdout sets.  
- **Model Build**: A Transformer-based architecture is instantiated and initialised.  
- **Training / Weight Loading**: The model can either be trained from scratch or initialised with pretrained weights
- **Evaluation and Visualisation**: Output at selected transect

In [ ]:
try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=True)
    import os
    import sys

    sys.path.append('/content/drive/MyDrive/Transformer-GPU')
    os.chdir('/content/drive/MyDrive/Transformer-GPU')

    !pip install whittaker_eilers

In [ ]:
# === Standard libraries ===
import warnings
import yaml

# === PyTorch ===
import torch

# === Data, models, utilities ===
from functions.load_data import FullModelData
from functions.model_object import *
from functions.plotting import *
from functions.misc import *

# === Plotting / statistics ===
import matplotlib.pyplot as plt

# --- Device setup ---
if torch.cuda.is_available():
    print(f"Connected to GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("GPU not available. Falling back to CPU.")
    device = "cpu"

# --- Warning filters ---
warnings.simplefilter("ignore", category=FutureWarning)
warnings.simplefilter("ignore", category=UserWarning)

# --- Plot style ---
plt.style.use("bmh")
plt.rcParams["font.family"] = "Times New Roman"

In [ ]:
# === Model Configuration and Data Setup ===
settings = yaml.safe_load(open("config/model_settings.yml", "r"))
settings["holdout_sites"] = settings["training_sites"] # example of has single transect, holdout data cannot be empty, this is essentially full model deployemnt

data = FullModelData(settings)
settings

In [ ]:
# === Model Initialization and Architecture Build ===
tft = Transformer25(data, settings)   # initialise model wrapper
tft.build_dataloaders()               # prepare training/validation data loaders
tft.build_model(None)                 # construct model architecture

(a) Load pretrained weights (returns a status object)

In [ ]:
model = tft.model
state_dict = torch.load("model_weights.pth", map_location=device)
status = model.load_state_dict(state_dict)
print(status)
model.to(device)

(b) Or: Train from scratch

In [ ]:
tft.train_model()
model = tft.model

In [ ]:
# === Evaluation and Visualisation ===
transect = "aus0052-0088"                 # site identifier
compare_outputs(transect, model, data, settings)  # generate figures + diagnostics